# INF2006 G12 — Task 3: Next-week stock price direction (FINAL candidate notebook)
This notebook is **self-contained**: load the existing Task 2 S3 tables, create past-only features, compare a few models, choose the probability cutoff using 2016 validation data, evaluate on 2017 test data once, save the final pipeline and verify reloading.

**How to run (EMR JupyterHub, PySpark kernel):**
1. Run all cells from the top (Kernel → Restart & Run All). The last cell will print *SKIPPED* because it is still the training session; that is expected.
2. Note the `RUN_ID` printed by the save cell. Then **Kernel → Restart** (a new Spark session starts) and run **only the last cell**. It reloads the saved model from S3 without retraining and compares all saved 2017 test predictions against a tolerance of 1e-9.
3. File → Save, then File → Download as .ipynb so the outputs are stored in the notebook.

NumPy is installed on every node by the `install-numpy.sh` bootstrap action.

**Meaning of class 1:** the stock closes higher after the next five trading sessions. In both notebooks, the model predicts a **class**, not an exact future price or numerical volatility. The probability score estimates the likelihood of class 1.

**Important:** Only use information from the current or earlier date as a feature. The test split is for the final one-time assessment, not repeated tuning.


In [1]:
# CELL 1 — Settings and tools.
# The bucket/path come from your successful V1 Direction notebook.
import time, json, datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

spark.conf.set("spark.sql.shuffle.partitions", "32")
BUCKET = "inf2006-g12-ml"
BASE = "s3a://" + BUCKET
INPUT = BASE + "/task2/model_ready_v1"
PRICES_PATH = BASE + "/task2/cleaned/prices"
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT = BASE + "/task3final/direction/" + RUN_ID
SEED = 42
SPLITS = ["training", "validation", "testing"]
BASE_FEATURES = ["daily_return", "return_5d", "volatility_20d", "volume_change_1d", "intraday_range"]
# These came from V1, which you already tested.
EXTRA_V1 = ["volatility_5d", "volatility_60d", "vol_ratio_5_60", "abs_return_5d",
            "range_5d", "return_20d", "volume_z_20d", "market_return_1d", "market_volatility_5d"]
# These are new: longer-term trends, relative strength, and market breadth.
NEW_FEATURES = ["return_10d", "return_60d", "price_to_ma20", "price_to_ma60",
                "price_in_range_20d", "return_vs_vol20", "relative_return_5d",
                "relative_return_20d", "market_avg_return_5d", "market_avg_return_20d",
                "market_breadth_1d", "market_breadth_5d"]
TREND_FEATURES = BASE_FEATURES + ["return_10d", "return_20d", "return_60d",
             "price_to_ma20", "price_to_ma60", "price_in_range_20d", "return_vs_vol20", "volatility_5d"]
MARKET_FEATURES = TREND_FEATURES + ["market_return_1d", "market_avg_return_5d",
                   "market_avg_return_20d", "market_breadth_1d", "market_breadth_5d",
                   "relative_return_5d", "relative_return_20d"]
FULL_FEATURES = list(dict.fromkeys(BASE_FEATURES + EXTRA_V1 + NEW_FEATURES))
print("Direction V2 | Spark", spark.version, "| RUN_ID", RUN_ID)
print("Input", INPUT)
print("Output", OUT)


Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
8,application_1795006229284_0009,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Direction V2 | Spark 3.5.5-amzn-1 | RUN_ID 20261118T150036Z
Input s3a://inf2006-g12-ml/task2/model_ready_v1
Output s3a://inf2006-g12-ml/task3final/direction/20261118T150036Z

In [2]:

def prepare(df):
    required = ["ticker", "trade_date", "label"] + BASE_FEATURES
    missing = sorted(set(required) - set(df.columns))
    assert not missing, "Missing input columns: " + str(missing)
    out = df.select("ticker", F.to_date("trade_date").alias("trade_date"),
         *[F.col(c).cast("double").alias(c) for c in BASE_FEATURES],
         F.col("label").cast("double").alias("target"))
    invalid = F.col("target").isNull() | (~F.col("target").isin(0.0, 1.0))
    for c in BASE_FEATURES:
        invalid = invalid | F.col(c).isNull() | F.isnan(c) | (F.abs(F.col(c)) == float("inf"))
    assert out.filter(invalid).limit(1).count() == 0, "Invalid target or original feature values"
    assert out.groupBy("ticker", "trade_date").count().filter("count > 1").limit(1).count() == 0, "Duplicate ticker-date"
    return out

data = {}
for split in SPLITS:
    data[split] = prepare(spark.read.parquet(INPUT + "/" + split)).cache()
    count = data[split].count()
    summ = data[split].agg(F.min("trade_date"), F.max("trade_date"), F.avg("target")).first()
    print(split, "rows", count, "dates", summ[0], "to", summ[1], "rise proportion", round(summ[2], 4))
expected = {"training": 349590, "validation": 122801, "testing": 126001}
for split in SPLITS:
    assert data[split].count() == expected[split], "Row count changed for " + split
print("Input validation passed.")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

training rows 349590 dates 2013-02-15 to 2015-12-23 rise proportion 0.5454
validation rows 122801 dates 2016-01-04 to 2016-12-22 rise proportion 0.5476
testing rows 126001 dates 2017-01-03 to 2017-12-29 rise proportion 0.5634
Input validation passed.

In [3]:
# Build indicators only from past/current trading dates.
# Example: return_60d compares today's closing price with 60 sessions ago.
# Example: market_breadth_1d = fraction of stocks that rose today.
prices = spark.read.parquet(PRICES_PATH).select(
    "ticker", F.to_date("trade_date").alias("trade_date"),
    *[F.col(c).cast("double").alias(c) for c in ["high_price", "low_price", "close_price", "volume"]])
assert prices.groupBy("ticker", "trade_date").count().filter("count > 1").limit(1).count() == 0, "Duplicate prices"
w = Window.partitionBy("ticker").orderBy("trade_date")
def history(days):
    return w.rowsBetween(-(days - 1), 0)
p = (prices.withColumn("prior_close", F.lag("close_price").over(w))
     .withColumn("ret", F.when(F.col("prior_close") > 0, F.col("close_price")/F.col("prior_close") - 1))
     .withColumn("rng", F.when(F.col("close_price") > 0, (F.col("high_price")-F.col("low_price"))/F.col("close_price")))
     .withColumn("logvol", F.when(F.col("volume") >= 0, F.log1p("volume"))))
p = (p.withColumn("volatility_5d", F.stddev_samp("ret").over(history(5)))
     .withColumn("volatility_60d", F.stddev_samp("ret").over(history(60)))
     .withColumn("abs_return_5d", F.avg(F.abs("ret")).over(history(5)))
     .withColumn("range_5d", F.avg("rng").over(history(5)))
     .withColumn("volume_z_20d", F.when(F.stddev_samp("logvol").over(history(20)) > 0,
         (F.col("logvol") - F.avg("logvol").over(history(20))) / F.stddev_samp("logvol").over(history(20)))))
for n in [10, 20, 60]:
    lag_price = F.lag("close_price", n).over(w)
    p = p.withColumn("return_" + str(n) + "d",
         F.when(lag_price > 0, F.col("close_price")/lag_price - 1))
p = (p.withColumn("vol_ratio_5_60", F.when(F.col("volatility_60d") > 0,
           F.col("volatility_5d")/F.col("volatility_60d")))
     .withColumn("price_to_ma20", F.when(F.avg("close_price").over(history(20)) > 0,
           F.col("close_price")/F.avg("close_price").over(history(20)) - 1))
     .withColumn("price_to_ma60", F.when(F.avg("close_price").over(history(60)) > 0,
           F.col("close_price")/F.avg("close_price").over(history(60)) - 1))
     .withColumn("price_in_range_20d", F.when(
           F.max("high_price").over(history(20)) > F.min("low_price").over(history(20)),
           (F.col("close_price")-F.min("low_price").over(history(20))) /
           (F.max("high_price").over(history(20))-F.min("low_price").over(history(20)))))
     .withColumn("return_vs_vol20", F.when(F.col("volatility_60d") > 0,
           F.col("return_20d")/F.col("volatility_60d"))))
# First compute the market's daily average, then rolling summaries of PREVIOUS days.
market = (p.groupBy("trade_date")
    .agg(F.avg("ret").alias("market_return_1d"),
         F.avg(F.when(F.col("ret").isNotNull(), (F.col("ret") > 0).cast("double"))).alias("market_breadth_1d"),
         F.avg("volatility_5d").alias("market_volatility_5d")))
mw = Window.orderBy("trade_date")
market = (market.withColumn("market_avg_return_5d", F.avg("market_return_1d").over(mw.rowsBetween(-4, 0)))
                .withColumn("market_avg_return_20d", F.avg("market_return_1d").over(mw.rowsBetween(-19, 0)))
                .withColumn("market_breadth_5d", F.avg("market_breadth_1d").over(mw.rowsBetween(-4, 0))))
p = p.join(market, "trade_date", "left")
# Compute the stock's own five-day return from cleaned prices, then subtract market trend.
lag5 = F.lag("close_price", 5).over(w)
p = (p.withColumn("stock_return_5d", F.when(lag5 > 0, F.col("close_price")/lag5 - 1))
      .withColumn("relative_return_5d", F.col("stock_return_5d")-5*F.col("market_avg_return_5d"))
      .withColumn("relative_return_20d", F.col("return_20d")-20*F.col("market_avg_return_20d")))
extra_cols = list(dict.fromkeys(EXTRA_V1 + NEW_FEATURES))
extra = p.select("ticker", "trade_date", *extra_cols)
for c in extra_cols:
    extra = extra.withColumn(c, F.when(F.isnan(c) | (F.abs(F.col(c)) == float("inf")),
                              F.lit(None).cast("double")).otherwise(F.col(c)))
for split in SPLITS:
    old = data[split]
    joined = old.join(extra, ["ticker", "trade_date"], "left").cache()
    assert joined.count() == old.count(), "Join changed row count"
    data[split] = joined
    old.unpersist()
print("Added", len(extra_cols), "historical features. V2 has", len(FULL_FEATURES), "total available features.")
print("Missing training values (filled from TRAINING medians only):")
data["training"].select(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in extra_cols]).show(vertical=True)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Added 21 historical features. V2 has 26 total available features.
Missing training values (filled from TRAINING medians only):
-RECORD 0----------------------
 volatility_5d         | 0     
 volatility_60d        | 0     
 vol_ratio_5_60        | 0     
 abs_return_5d         | 0     
 range_5d              | 0     
 return_20d            | 7425  
 volume_z_20d          | 0     
 market_return_1d      | 0     
 market_volatility_5d  | 0     
 return_10d            | 2475  
 return_60d            | 27166 
 price_to_ma20         | 0     
 price_to_ma60         | 0     
 price_in_range_20d    | 0     
 return_vs_vol20       | 7425  
 relative_return_5d    | 0     
 relative_return_20d   | 7425  
 market_avg_return_5d  | 0     
 market_avg_return_20d | 0     
 market_breadth_1d     | 0     
 market_breadth_5d     | 0

In [4]:

print("Has return_10d:",
      "return_10d" in data["training"].columns)

missing = [
    c for c in FULL_FEATURES
    if c not in data["training"].columns
]

print("Missing features:", missing)

assert not missing, "Feature engineering incomplete"

print("FEATURE CHECK PASSED")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Has return_10d: True
Missing features: []
FEATURE CHECK PASSED

In [5]:
#  Metrics and a simple baseline.
# AUC and balanced accuracy are the two headline numbers for report
auc_evaluator = BinaryClassificationEvaluator(labelCol="target", rawPredictionCol="risk_score", metricName="areaUnderROC")
def with_cutoff(df, cutoff):
    return df.withColumn("prediction", (F.col("risk_score") >= F.lit(float(cutoff))).cast("double"))
def metric(df, title):
    counts = {(int(r["target"]), int(r["prediction"])): r["count"]
              for r in df.groupBy("target", "prediction").count().collect()}
    tn, fp, fn, tp = [counts.get(k, 0) for k in [(0,0), (0,1), (1,0), (1,1)]]
    n = tn + fp + fn + tp
    sensitivity = tp/(tp+fn) if tp+fn else 0
    specificity = tn/(tn+fp) if tn+fp else 0
    precision = tp/(tp+fp) if tp+fp else 0
    f1 = 2*precision*sensitivity/(precision+sensitivity) if precision+sensitivity else 0
    result = dict(name=title, n=n, auc=round(auc_evaluator.evaluate(df), 5),
        balanced_accuracy=round((sensitivity+specificity)/2, 5),
        accuracy=round((tp+tn)/n, 5), f1=round(f1, 5),
        recall=round(sensitivity, 5), specificity=round(specificity, 5),
        tn=tn, fp=fp, fn=fn, tp=tp)
    print(title, result)
    return result
# Search for the decision cutoff using tiny aggregated histogram (not 100 full-data queries).
def best_cutoff(scores):
    hist = (scores.select(F.least(F.greatest(F.floor(F.col("risk_score")*100), F.lit(0)), F.lit(100)).cast("int").alias("bin"), "target")
         .groupBy("bin", "target").count().collect())
    positives, negatives = [0]*101, [0]*101
    for row in hist:
        (positives if row["target"] == 1.0 else negatives)[row["bin"]] += row["count"]
    P, N = sum(positives), sum(negatives)
    assert P and N
    best = (-1, 0.5)
    for k in range(10, 91):
        tp, tn = sum(positives[k:]), sum(negatives[:k])
        bal = 0.5*(tp/P+tn/N)
        if bal > best[0]: best = (bal, k/100.0)
    return best[1]
majority = float(data["training"].groupBy("target").count().orderBy(F.desc("count")).first()["target"])
print("Training majority class:", majority)
print("Chance-level AUC = 0.50, balanced accuracy = 0.50.")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Training majority class: 1.0
Chance-level AUC = 0.50, balanced accuracy = 0.50.

In [6]:
# Train
# We compare stronger regularization and different groups of indicators.
RUN_GBT = True  # Change to False if tree training is too slow.

def pipeline_for(kind, cols, strength=0.1):
    filled = [c+"_filled" for c in cols]
    stages = [Imputer(inputCols=cols, outputCols=filled, strategy="median")]
    if kind == "lr":
        stages += [VectorAssembler(inputCols=filled, outputCol="unscaled"),
                   StandardScaler(inputCol="unscaled", outputCol="features", withStd=True, withMean=False),
                   LogisticRegression(featuresCol="features", labelCol="target", maxIter=120,
                                      regParam=strength, elasticNetParam=0.0)]
    else:
        stages += [VectorAssembler(inputCols=filled, outputCol="features"),
                   GBTClassifier(featuresCol="features", labelCol="target", maxIter=35,
                                 maxDepth=3, minInstancesPerNode=60, seed=SEED)]
    return Pipeline(stages=stages)

candidates = [
    ("logistic_base_r01", "lr", BASE_FEATURES, 0.1),
    ("logistic_trend_r01", "lr", TREND_FEATURES, 0.1),
    ("logistic_market_r01", "lr", MARKET_FEATURES, 0.1),
    ("logistic_market_r10", "lr", MARKET_FEATURES, 1.0),
    ("logistic_full_r01", "lr", FULL_FEATURES, 0.1),
    ("logistic_full_r10", "lr", FULL_FEATURES, 1.0),
]
if RUN_GBT:
    candidates += [("gbt_market", "gbt", MARKET_FEATURES, None)]

fitted, results = {}, []
for name, kind, cols, strength in candidates:
    start = time.time()
    print("\nTraining", name, "with", len(cols), "features", flush=True)
    model = pipeline_for(kind, cols, strength).fit(data["training"])
    # We use two periods within 2016 as a stability check: early and late 2016.
    scores = (model.transform(data["validation"])
        .withColumn("risk_score", vector_to_array("probability")[1])
        .select("trade_date", "target", "risk_score").cache())
    scores.count()
    cutoff = best_cutoff(scores)
    pred = with_cutoff(scores, cutoff)
    full = metric(pred, name + "_2016")
    early = pred.filter(F.col("trade_date") < F.lit("2016-07-01"))
    late = pred.filter(F.col("trade_date") >= F.lit("2016-07-01"))
    auc_early = round(auc_evaluator.evaluate(early), 5)
    auc_late = round(auc_evaluator.evaluate(late), 5)
    print("2016 AUC: Jan-Jun", auc_early, "| Jul-Dec", auc_late, "| cutoff", cutoff)
    results.append(dict(name=name, auc=full["auc"], balanced_accuracy=full["balanced_accuracy"],
                        cutoff=cutoff, auc_early=auc_early, auc_late=auc_late,
                        worst_half_auc=min(auc_early, auc_late), seconds=round(time.time()-start, 1)))
    fitted[name] = model
    scores.unpersist()
# Prefer consistency: highest worst-half AUC first, then overall 2016 AUC.
# This helps avoid selecting a model that only happened to work in one period.
chosen = sorted(results, key=lambda r: (-r["worst_half_auc"], -r["auc"], r["name"]))[0]
chosen_name, chosen_cutoff = chosen["name"], chosen["cutoff"]
final_model = fitted[chosen_name]
chosen_features = next(cols for name, kind, cols, _ in candidates if name == chosen_name)
print("\n================ MODEL SELECTED FROM 2016 ================")
for r in results:
    print(r["name"], "AUC", r["auc"], "balanced accuracy", r["balanced_accuracy"],
          "worst half AUC", r["worst_half_auc"], "seconds", r["seconds"])
print("WINNER:", chosen_name, "| 2016 AUC", chosen["auc"],
      "| 2016 balanced accuracy", chosen["balanced_accuracy"],
      "| cutoff", chosen_cutoff)
print("No 2017 data used to select the winner.")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…


Training logistic_base_r01 with 5 features
logistic_base_r01_2016 {'name': 'logistic_base_r01_2016', 'n': 122801, 'auc': 0.53094, 'balanced_accuracy': 0.52586, 'accuracy': 0.53887, 'f1': 0.61141, 'recall': 0.66246, 'specificity': 0.38927, 'tn': 21625, 'fp': 33928, 'fn': 22699, 'tp': 44549}
2016 AUC: Jan-Jun 0.51425 | Jul-Dec 0.54274 | cutoff 0.54

Training logistic_trend_r01 with 13 features
logistic_trend_r01_2016 {'name': 'logistic_trend_r01_2016', 'n': 122801, 'auc': 0.51837, 'balanced_accuracy': 0.51625, 'accuracy': 0.50812, 'f1': 0.48967, 'recall': 0.43093, 'specificity': 0.60157, 'tn': 33419, 'fp': 22134, 'fn': 38269, 'tp': 28979}
2016 AUC: Jan-Jun 0.50389 | Jul-Dec 0.5316 | cutoff 0.55

Training logistic_market_r01 with 20 features
logistic_market_r01_2016 {'name': 'logistic_market_r01_2016', 'n': 122801, 'auc': 0.49833, 'balanced_accuracy': 0.51037, 'accuracy': 0.49417, 'f1': 0.42418, 'recall': 0.34022, 'specificity': 0.68052, 'tn': 37805, 'fp': 17748, 'fn': 44369, 'tp': 22879

In [7]:
# Feature importance explains what the model paid attention to.
# Large absolute logistic weights = stronger association, NOT guaranteed prediction.
stage = final_model.stages[-1]
weights = stage.coefficients.toArray() if hasattr(stage, "coefficients") else stage.featureImportances.toArray()
print("Winner:", chosen_name)
for col, weight in sorted(zip(chosen_features, weights), key=lambda x:-abs(x[1])):
    print("{:<28} {: .5f}".format(col, float(weight)))


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Winner: logistic_full_r01
market_volatility_5d          0.12300
market_avg_return_20d        -0.06336
price_to_ma60                -0.04742
volatility_60d               -0.03475
relative_return_20d           0.03022
volatility_20d                0.02759
return_60d                    0.02590
market_return_1d             -0.02298
market_avg_return_5d         -0.02179
market_breadth_5d             0.02073
price_in_range_20d           -0.02047
relative_return_5d            0.01648
market_breadth_1d            -0.01633
price_to_ma20                -0.01571
intraday_range                0.01528
range_5d                     -0.01463
volatility_5d                -0.01434
daily_return                 -0.01293
volume_z_20d                 -0.00871
return_20d                    0.00624
return_10d                   -0.00598
abs_return_5d                 0.00312
return_5d                     0.00255
volume_change_1d             -0.00214
return_vs_vol20               0.00130
vol_ratio_5_60          

In [8]:
# Final 2017 test. Run this after selecting the model from 2016.

test_scores = (final_model.transform(data["testing"])
    .withColumn("risk_score", vector_to_array("probability")[1]))
test_predictions = with_cutoff(test_scores, chosen_cutoff).cache()
test_predictions.count()
final_test = metric(test_predictions, chosen_name + "_FINAL_TEST_2017")
majority_predictions = (data["testing"]
    .withColumn("risk_score", F.lit(majority))
    .withColumn("prediction", F.lit(majority)))
baseline_test = metric(majority_predictions, "always_training_majority_2017")
print("\nCONFUSION MATRIX: actual rows / predicted columns")
print("              predicted 0   predicted 1")
print("actual 0      {:>10}    {:>10}".format(final_test["tn"], final_test["fp"]))
print("actual 1      {:>10}    {:>10}".format(final_test["fn"], final_test["tp"]))
print("\nSUMMARY | validation 2016: AUC", chosen["auc"], "balanced accuracy", chosen["balanced_accuracy"],
      "| test 2017: AUC", final_test["auc"], "balanced accuracy", final_test["balanced_accuracy"])
print("Baseline (always predict training majority) 2017:", baseline_test)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

logistic_full_r01_FINAL_TEST_2017 {'name': 'logistic_full_r01_FINAL_TEST_2017', 'n': 126001, 'auc': 0.51324, 'balanced_accuracy': 0.51055, 'accuracy': 0.49199, 'f1': 0.44687, 'recall': 0.36423, 'specificity': 0.65686, 'tn': 36134, 'fp': 18876, 'fn': 45134, 'tp': 25857}
always_training_majority_2017 {'name': 'always_training_majority_2017', 'n': 126001, 'auc': 0.5, 'balanced_accuracy': 0.5, 'accuracy': 0.56342, 'f1': 0.72075, 'recall': 1.0, 'specificity': 0.0, 'tn': 0, 'fp': 55010, 'fn': 0, 'tp': 70991}

CONFUSION MATRIX: actual rows / predicted columns
              predicted 0   predicted 1
actual 0           36134         18876
actual 1           45134         25857

SUMMARY | validation 2016: AUC 0.55605 balanced accuracy 0.54744 | test 2017: AUC 0.51324 balanced accuracy 0.51055
Baseline (always predict training majority) 2017: {'name': 'always_training_majority_2017', 'n': 126001, 'auc': 0.5, 'balanced_accuracy': 0.5, 'accuracy': 0.56342, 'f1': 0.72075, 'recall': 1.0, 'specificity

In [9]:
# Save winning model and its predictions to a NEW S3 folder.
MODEL_PATH = OUT + "/model"
PRED_PATH = OUT + "/predictions"
METRICS_PATH = OUT + "/metrics"
SETTINGS_PATH = OUT + "/settings"
TEST_FEATURES_PATH = OUT + "/test_features"
final_model.write().overwrite().save(MODEL_PATH)
(test_predictions.select("ticker", "trade_date", "target", "prediction", "risk_score")
    .write.mode("overwrite").parquet(PRED_PATH))
# Raw (un-imputed) 2017 inputs, so a fresh session can re-score them with the saved model only.
data["testing"].select("ticker", "trade_date", "target", *chosen_features) \
    .write.mode("overwrite").parquet(TEST_FEATURES_PATH)
records = [dict(stage="validation_2016", **r) for r in results] + [dict(stage="test_2017", **final_test)]
spark.createDataFrame([(json.dumps(r),) for r in records], ["value"]).coalesce(1).write.mode("overwrite").text(METRICS_PATH)
metadata = dict(run_id=RUN_ID, task="direction", version="v2", chosen_model=chosen_name,
                cutoff=chosen_cutoff, features=chosen_features, model_path=MODEL_PATH,
                input_path=INPUT, prices_path=PRICES_PATH, predictions_path=PRED_PATH, test_features_path=TEST_FEATURES_PATH,
    spark_application_id=spark.sparkContext.applicationId)
spark.createDataFrame([(json.dumps(metadata),)], ["value"]).coalesce(1).write.mode("overwrite").text(SETTINGS_PATH)
print("SAVED MODEL:", MODEL_PATH)
print("SAVED PREDICTIONS:", PRED_PATH)
print("SETTINGS:", SETTINGS_PATH)
print("TEST FEATURES:", TEST_FEATURES_PATH)
print("RUN_ID:", RUN_ID, "| Spark application:", spark.sparkContext.applicationId)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SAVED MODEL: s3a://inf2006-g12-ml/task3final/direction/20261118T150036Z/model
SAVED PREDICTIONS: s3a://inf2006-g12-ml/task3final/direction/20261118T150036Z/predictions
SETTINGS: s3a://inf2006-g12-ml/task3final/direction/20261118T150036Z/settings
TEST FEATURES: s3a://inf2006-g12-ml/task3final/direction/20261118T150036Z/test_features
RUN_ID: 20261118T150036Z | Spark application: application_1795006229284_0009

## Fresh-session reload check
**Kernel → Restart first**, then run only the cell below. It must show a different Spark application ID from the training run.

In [1]:
# FRESH-SESSION RELOAD CHECK (project brief 3.3)
# Run this cell ONLY after Kernel -> Restart. Nothing is retrained: it loads the saved
# PipelineModel from S3, re-scores the saved 2017 test inputs, and compares EVERY row
# with the predictions saved by the training session.
import json, datetime
from pyspark.sql import functions as F
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array

BUCKET = "inf2006-g12-ml"   # same bucket as cell 1 (this cell runs alone after a kernel restart)
TASK_DIR = "s3a://" + BUCKET + "/task3final/direction"  # one sub-folder per RUN_ID
RUN_ID = ""          # blank = newest saved run; or paste the RUN_ID printed by the save cell
TOLERANCE = 1e-9     # maximum allowed absolute difference in class-1 probability

Path = spark._jvm.org.apache.hadoop.fs.Path
root = Path(TASK_DIR)
fs = root.getFileSystem(spark._jsc.hadoopConfiguration())
if not RUN_ID:
    runs = sorted(st.getPath().getName() for st in fs.listStatus(root)
                  if st.isDirectory() and fs.exists(Path(TASK_DIR + "/" + st.getPath().getName() + "/test_features")))
    assert runs, "No saved run with test_features under " + TASK_DIR + " - run the notebook from the top first."
    RUN_ID = runs[-1]
RUN = TASK_DIR + "/" + RUN_ID
cfg = json.loads(spark.read.text(RUN + "/settings").first()["value"])
cutoff = float(cfg.get("cutoff", cfg.get("threshold")))
this_app = spark.sparkContext.applicationId
print("RUN_ID:", RUN_ID, "| trained in Spark app", cfg.get("spark_application_id"), "| this session:", this_app)

if cfg.get("spark_application_id") == this_app:
    print("SKIPPED: this is still the training session. Do Kernel -> Restart, then run ONLY this cell.")
else:
    for part in ["model", "test_features", "predictions", "settings"]:
        assert fs.exists(Path(RUN + "/" + part)), "MISSING saved files: " + RUN + "/" + part
    model = PipelineModel.load(RUN + "/model")          # loaded, not fitted
    print("Loaded stages:", [type(s).__name__ for s in model.stages])
    features = spark.read.parquet(RUN + "/test_features")
    saved = spark.read.parquet(RUN + "/predictions")
    new = (model.transform(features)
           .withColumn("risk_new", vector_to_array("probability")[1])
           .withColumn("pred_new", (F.col("risk_new") >= F.lit(cutoff)).cast("double"))
           .select("ticker", "trade_date", "risk_new", "pred_new"))
    both = saved.join(new, ["ticker", "trade_date"], "full_outer")
    s = both.agg(F.count(F.lit(1)).alias("rows_compared"),
        F.sum(F.col("risk_score").isNull().cast("int")).alias("rows_only_in_reload"),
        F.sum(F.col("risk_new").isNull().cast("int")).alias("rows_only_in_saved"),
        F.max(F.abs(F.col("risk_score") - F.col("risk_new"))).alias("max_abs_prob_diff"),
        F.sum((F.col("prediction") != F.col("pred_new")).cast("int")).alias("class_mismatches")).first().asDict()
    s["saved_prediction_rows"] = saved.count()
    passed = (s["rows_only_in_reload"] == 0 and s["rows_only_in_saved"] == 0 and s["class_mismatches"] == 0
              and s["max_abs_prob_diff"] is not None and s["max_abs_prob_diff"] <= TOLERANCE)
    record = dict(check="fresh-session saved-model reload", task=cfg.get("task"), run_id=RUN_ID,
                  model_path=RUN + "/model", cutoff=cutoff, tolerance=TOLERANCE,
                  checked_at_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
                  training_spark_app=cfg.get("spark_application_id"), reload_spark_app=this_app,
                  passed=passed, **s)
    spark.createDataFrame([(json.dumps(record, default=str),)], ["value"]).coalesce(1) \
         .write.mode("overwrite").text(RUN + "/reload_check")
    print(json.dumps(record, indent=1, default=str))
    print("\nRELOAD CHECK", "PASSED" if passed else "FAILED", "-", s["saved_prediction_rows"],
          "test rows compared, tolerance", TOLERANCE, "| record saved to", RUN + "/reload_check")


Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
9,application_1795006229284_0010,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

RUN_ID: 20261118T150036Z | trained in Spark app application_1795006229284_0009 | this session: application_1795006229284_0010
Loaded stages: ['ImputerModel', 'VectorAssembler', 'StandardScalerModel', 'LogisticRegressionModel']
{
 "check": "fresh-session saved-model reload",
 "task": "direction",
 "run_id": "20261118T150036Z",
 "model_path": "s3a://inf2006-g12-ml/task3final/direction/20261118T150036Z/model",
 "cutoff": 0.54,
 "tolerance": 1e-09,
 "checked_at_utc": "2026-11-18T15:05:53+00:00",
 "training_spark_app": "application_1795006229284_0009",
 "reload_spark_app": "application_1795006229284_0010",
 "passed": true,
 "rows_compared": 126001,
 "rows_only_in_reload": 0,
 "rows_only_in_saved": 0,
 "max_abs_prob_diff": 0.0,
 "class_mismatches": 0,
 "saved_prediction_rows": 126001
}

RELOAD CHECK PASSED - 126001 test rows compared, tolerance 1e-09 | record saved to s3a://inf2006-g12-ml/task3final/direction/20261118T150036Z/reload_check